# MedleyVox One Click Gradio

แยกเสียงนักร้องหลายคนในเพลงเดียว กดรันเซลล์ด้านล่างเซลล์เดียว แล้วรอจนขึ้นลิงก์ Gradio

In [ ]:
# @title กดรันครั้งเดียวเพื่อเปิด MedleyVox (แยกเสียงนักร้องหลายคน)
# @markdown ต้องเลือกเครื่องแบบ GPU ก่อน: เมนู Runtime > Change runtime type > T4 GPU
language = "th_TH" # @param ["th_TH", "en_US"]

from IPython.display import clear_output
from pathlib import Path
import json, os

repo_path = Path('/content/MedleyVox-Inference-WebUI')
py = '/content/mvenv/bin/python'

%cd /content
if repo_path.exists():
    %cd /content/MedleyVox-Inference-WebUI
    !git pull -q
else:
    !git clone -q https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI
    %cd /content/MedleyVox-Inference-WebUI

import subprocess, sys

def run(step, cmd):
    # รันทีละขั้น ถ้าพังให้หยุดและแสดงสาเหตุ (ไม่ลบข้อความทิ้ง)
    print(f'▶ {step} ...')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise SystemExit(f'❌ ขั้น "{step}" ไม่สำเร็จ — ส่งข้อความด้านบนให้คนดูแลได้เลย')

if not Path('/content/mvenv/.ok').exists():
    print('กำลังติดตั้งของจำเป็น (ครั้งแรกใช้เวลาประมาณ 10-15 นาที) ...')
    # fairseq ติดตั้งบน Python ของ Colab (3.12) ไม่ได้ จึงแยกใช้ Python 3.10
    run('ติดตั้งตัวช่วย uv', f'{sys.executable} -m pip install -q uv')
    run('สร้าง Python 3.10', f'rm -rf /content/mvenv && {sys.executable} -m uv venv --seed -p 3.10 /content/mvenv')
    run('เตรียม pip', f'{py} -m pip install -q "pip<24.1" setuptools wheel cython numpy')
    run('ติดตั้ง torch', f'{py} -m pip install -q "torch<2.6.0" torchaudio --index-url https://download.pytorch.org/whl/cu124')
    run('ติดตั้งส่วนประกอบ MedleyVox (นานสุด)', f'{py} -m pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124')
    run('ตรวจว่าติดตั้งครบ', f'{py} -c "import torch, fairseq, asteroid, gradio; print(torch.cuda.is_available())"')
    Path('/content/mvenv/.ok').touch()
else:
    print('ติดตั้งไว้แล้ว ข้ามขั้นติดตั้ง')

print('กำลังดาวน์โหลดโมเดล ...')
models = {
    'vocals_238': 'vocals%20238',
    'multi_singing_librispeech_138': 'multi_singing_librispeech_138',
    'singing_librispeech_ft_iSRNet': 'singing_librispeech_ft_iSRNet',
}
for name, remote in models.items():
    folder = repo_path / 'checkpoints' / name
    folder.mkdir(parents=True, exist_ok=True)
    for f in ['vocals.pth', 'vocals.json']:
        if not (folder / f).exists():
            !wget -q -O "{folder}/{f}" "https://huggingface.co/Cyru5/MedleyVox/resolve/main/{remote}/{f}"
(repo_path / 'pretrained').mkdir(exist_ok=True)
if not (repo_path / 'pretrained' / 'xlsr_53_56k.pt').exists():
    !wget -q -O pretrained/xlsr_53_56k.pt https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt
for d in ['inputs', 'results']:
    (repo_path / d).mkdir(exist_ok=True)

# ไฟล์ภาษาไทย (ต้นฉบับไม่มี จึงใส่มาในโน้ตบุ๊กนี้)
th_TH = {
 "Input audio files to inference": "แยกเสียงจากไฟล์เสียง",
 "Input folder to inference": "แยกเสียงจากโฟลเดอร์",
 "Please upload at least one audio file!": "กรุณาอัปโหลดไฟล์เสียงอย่างน้อย 1 ไฟล์",
 "Please download the pretrained model xlsr_53_56k.pt from https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt and put it in the 'pretrained' folder!": "กรุณาดาวน์โหลดโมเดล xlsr_53_56k.pt จาก https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt แล้วนำไปไว้ในโฟลเดอร์ 'pretrained'",
 "Please select a model!": "กรุณาเลือกโมเดล",
 "Please select an output folder!": "กรุณาเลือกโฟลเดอร์ที่จะเก็บผลลัพธ์",
 "Please select an input folder!": "กรุณาเลือกโฟลเดอร์ไฟล์ต้นฉบับ",
 "Inference process finished.": "แยกเสียงเสร็จแล้ว",
 "Inference process stopped by user.": "หยุดการแยกเสียงแล้ว",
 "No active inference process.": "ไม่มีงานแยกเสียงที่กำลังทำอยู่",
 "Medley Vox is a [dataset for testing algorithms for separating multiple singers](https://arxiv.org/pdf/2211.07302) within a single music track. Also, the [authors of Medley Vox](https://github.com/jeonchangbin49/MedleyVox) proposed a neural network architecture for separating singers. However, unfortunately, they did not publish the weights. Later, their training process was [repeated by Cyru5](https://huggingface.co/Cyru5/MedleyVox/tree/main), who trained several models and published the weights in the public domain. Now this WebUI is created to use the trained models and weights for inference. Here are some precautions:<br>1. Put the [downloaded models](https://huggingface.co/Cyru5/MedleyVox) in the 'checkpoints' folder in folder format, with each model folder containing a model file (.pth) and its corresponding configuration file (.json).<br>2. If you use overlapadd and the choice of model is 'w2v' or 'w2v_chunk', you need to download the pretrained model [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) and put it in the 'pretrained' folder.<br>3. At present, the audio output sampling rate supported by the model is 24000kHz and cannot be changed. To solve this, you can use [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo), or [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) for audio super-resolution.<br>4. When using WebUI on cloud platforms or Colab, please place the audio to be processed in the 'inputs' folder, and the processing results will be stored in the 'results' folder. The 'Select folder' and 'Open folder' buttons are invalid in the cloud.": "Medley Vox คือ[ชุดข้อมูลสำหรับทดสอบวิธีแยกเสียงนักร้องหลายคน](https://arxiv.org/pdf/2211.07302)ที่ร้องอยู่ในเพลงเดียวกัน [ทีมผู้พัฒนา Medley Vox](https://github.com/jeonchangbin49/MedleyVox) ได้เสนอโครงข่ายประสาทเทียมสำหรับแยกเสียงนักร้องไว้ แต่ไม่ได้เผยแพร่ไฟล์โมเดลที่ฝึกแล้ว ต่อมา [Cyru5 ได้ฝึกโมเดลตามวิธีเดิมซ้ำ](https://huggingface.co/Cyru5/MedleyVox/tree/main) และเผยแพร่ให้ใช้ได้ฟรี หน้าเว็บนี้สร้างขึ้นเพื่อใช้โมเดลเหล่านั้นแยกเสียง ข้อควรทราบมีดังนี้<br>1. นำ[โมเดลที่ดาวน์โหลดมา](https://huggingface.co/Cyru5/MedleyVox)ไปไว้ในโฟลเดอร์ 'checkpoints' โดยแต่ละโมเดลอยู่ในโฟลเดอร์ของตัวเอง ภายในมีไฟล์โมเดล (.pth) และไฟล์ตั้งค่า (.json) คู่กัน<br>2. หากใช้ overlapadd แบบ 'w2v' หรือ 'w2v_chunk' ต้องดาวน์โหลดโมเดล [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) ไปไว้ในโฟลเดอร์ 'pretrained'<br>3. ขณะนี้โมเดลส่งออกเสียงที่อัตราสุ่ม 24000 Hz เท่านั้น เปลี่ยนไม่ได้ หากต้องการเสียงคมชัดขึ้น ใช้ [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo) หรือ [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) ช่วยเพิ่มความละเอียดเสียงได้<br>4. หากใช้บนคลาวด์หรือ Colab ให้นำไฟล์เสียงไปไว้ในโฟลเดอร์ 'inputs' แล้วผลลัพธ์จะอยู่ในโฟลเดอร์ 'results' ปุ่ม 'เลือกโฟลเดอร์' และ 'เปิดโฟลเดอร์' ใช้งานบนคลาวด์ไม่ได้",
 "Common options": "ตั้งค่าทั่วไป",
 "Select Model": "เลือกโมเดล",
 "Select which model you want to use.": "เลือกโมเดลที่ต้องการใช้",
 "Use overlapadd": "วิธีแบ่งช่วงประมวลผล (overlapadd)",
 "Use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.<br>If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use 'use_overlapadd'. Among the 'use_overlapadd' options, 'ola', 'ola_norm', and 'w2v' all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with 'vad_method' options spec and webrtc when using either of the '_chunk' methods. Chunking has proven to be very useful therefore it is on by default.": "ola, ola_norm และ w2v ใช้ค่าความยาวช่วง (ola_window_len) และระยะเลื่อน (ola_hop_len) ส่วน w2v_chunk และ sf_chunk แบ่งช่วงตามจุดที่มีเสียงร้อง (VAD) จึงต้องเลือกวิธี VAD ด้วย หากใช้ sf_chunk ต้องเลือกลักษณะเสียง (spectral features) ด้วย<br>หากไฟล์ยาวเกินไป หน่วยความจำการ์ดจอ (VRAM) อาจไม่พอ ให้ใช้ตัวเลือกนี้ช่วย โดย 'ola', 'ola_norm' และ 'w2v' ให้ผลดีทั้งหมด หากไม่ได้ผลค่อยลอง w2v_chunk หรือ sf_chunk และลองสลับวิธี VAD ระหว่าง spec กับ webrtc ได้ ค่าเริ่มต้นเปิดการแบ่งช่วงไว้เพราะช่วยได้มาก",
 "Use GPU": "ใช้การ์ดจอ (GPU)",
 "Use GPU for inference.": "ใช้การ์ดจอช่วยประมวลผล (เร็วกว่า)",
 "Save results in separate folders": "เก็บผลลัพธ์แยกโฟลเดอร์",
 "Save results in separate folders with the same name as the input file.": "เก็บผลลัพธ์ของแต่ละไฟล์ในโฟลเดอร์ชื่อเดียวกับไฟล์ต้นฉบับ",
 "Skip error files": "ข้ามไฟล์ที่มีปัญหา",
 "Skip error files while separating instead of stopping.": "หากไฟล์ใดมีปัญหา ให้ข้ามไปทำไฟล์ถัดไปแทนการหยุดทั้งหมด",
 "Output format": "ชนิดไฟล์ผลลัพธ์",
 "Select the output format.": "เลือกชนิดไฟล์เสียงที่ต้องการ",
 "[Click to expand] Advanced options": "[กดเพื่อเปิด] ตั้งค่าขั้นสูง",
 "VAD method": "วิธีตรวจหาช่วงที่มีเสียงร้อง (VAD)",
 "What method do you want to use for 'voice activity detection (vad) -- split chunks -- processing. Only valid when 'w2v_chunk' or 'sf_chunk' for args.use_overlapadd.": "วิธีหาช่วงที่มีเสียงร้องเพื่อแบ่งไฟล์ ใช้ได้เฉพาะเมื่อเลือก 'w2v_chunk' หรือ 'sf_chunk'",
 "Spectral features": "ลักษณะเสียงที่ใช้เทียบ (spectral features)",
 "What spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)": "ลักษณะเสียงที่ใช้เทียบว่าช่วงไหนเป็นนักร้องคนไหน (ใช้ได้เฉพาะ sf_chunk)",
 "OLA window length": "ความยาวช่วง OLA",
 "OLA window size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).": "ความยาวแต่ละช่วงเป็นวินาที ใช้ได้เฉพาะ ola หรือ ola_norm ใส่ 0 = ใช้ค่าเริ่มต้น",
 "OLA hop length": "ระยะเลื่อน OLA",
 "OLA hop size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).": "ระยะเลื่อนแต่ละช่วงเป็นวินาที ใช้ได้เฉพาะ ola หรือ ola_norm ใส่ 0 = ใช้ค่าเริ่มต้น",
 "Wav2Vec nth layer output": "ชั้นผลลัพธ์ของ Wav2Vec",
 "Wav2Vec nth layer output, only valid when using w2v or w2v_chunk. For example: 0 1 2 3, default: 0": "เลขชั้นของ Wav2Vec ที่ใช้ ใช้ได้เฉพาะ w2v หรือ w2v_chunk เช่น 0 1 2 3 ค่าเริ่มต้น 0",
 "Use EMA model": "ใช้โมเดลแบบ EMA",
 "Use EMA model or online model? Only vaind when args.ema it True (model trained with EMA).": "ใช้โมเดลแบบ EMA หรือแบบปกติ ใช้ได้เฉพาะโมเดลที่ฝึกแบบ EMA",
 "Mix consistent output": "ปรับผลรวมให้ตรงต้นฉบับ",
 "Only valid when the model is trained with mixture_consistency loss.": "ใช้ได้เฉพาะโมเดลที่ฝึกด้วย mixture_consistency loss",
 "Reorder chunks": "เรียงช่วงใหม่",
 "OLA reorder chunks. Only valid when using ola or ola_norm.": "เรียงลำดับช่วงเสียงใหม่ ใช้ได้เฉพาะ ola หรือ ola_norm",
 "Input audio files": "ไฟล์เสียง",
 "Input one or more audio files": "ใส่ไฟล์เสียง 1 ไฟล์ขึ้นไป",
 "Input folder path": "ที่อยู่โฟลเดอร์ต้นฉบับ",
 "Audio files in the folder will be used for inference.": "ไฟล์เสียงทั้งหมดในโฟลเดอร์นี้จะถูกนำไปแยกเสียง",
 "Select folder": "เลือกโฟลเดอร์",
 "Open folder": "เปิดโฟลเดอร์",
 "Output folder path": "ที่อยู่โฟลเดอร์ผลลัพธ์",
 "Audio files will be saved in this folder.": "ไฟล์ที่แยกเสร็จจะเก็บไว้ในโฟลเดอร์นี้",
 "Output message": "ข้อความแจ้ง",
 "Forced stop inference": "หยุดการแยกเสียง"
}
(repo_path / 'utils' / 'locale' / 'th_TH.json').write_text(json.dumps(th_TH, ensure_ascii=False, indent=4), encoding='utf-8')
os.environ['LANGUAGE'] = language

clear_output()
print('เปิดหน้าเว็บ... รอจนขึ้นลิงก์ https://xxxx.gradio.live แล้วกดลิงก์นั้น')
print('ไฟล์เสียง: อัปโหลดในหน้าเว็บได้เลย ผลลัพธ์อยู่ในโฟลเดอร์ results (ดูได้จากแถบไฟล์ซ้ายมือ)')
!{py} webui.py --share
